# 09 - Delaunay Triangulations

        Source span: printed pages 191-218; PDF pages 202-229. This notebook is original standalone study material for *Computational Geometry: Algorithms and Applications*, Third Edition. It uses the book as orientation for chapter structure and concepts, but it does not copy textbook prose, exercises, screenshots, page crops, or figures.

        ## Standalone Goal

        The chapter question is: how do we turn interpolating terrain heights from sample locations into a precise geometric algorithm that can be inspected, tested, and implemented? The answer is not just a theorem statement. It is a chain of modeling decisions: choose the geometric primitive, identify the invariant that makes the primitive useful, pick a data structure that exposes only the necessary local information, and verify that the resulting algorithm reports the same answer as a slower direct method on small examples.

        This notebook teaches planar triangulations, Delaunay empty circle property, lifting, randomized construction. The diagrams are not decoration; each one is a small laboratory. The first visual fixes the geometric object of study. The second visual exposes the algorithmic state that changes over time. The computational check at the end records the invariant that should survive those state changes. When an algorithm is randomized, the notebook uses a deterministic seed so the displayed run is reproducible while still showing what changes under a random order. When an algorithm is sensitive to degeneracy, the notebook names the fragile predicate instead of hiding it inside a library call.

        A recurring theme in computational geometry is that the obvious mathematical definition is often too large to compute directly. A convex hull is an intersection of all convex sets, but the algorithm works with turns in a sorted list of points. A subdivision may be a planar set, but an overlay algorithm needs vertex, edge, face, and incidence records. A nearest-site region is defined by infinitely many distances, but a diagram is built from bisectors and events. For this chapter, the same translation happens through Delaunay triangulation as the projection of a lower convex hull. The notebook therefore keeps two views in sync: the continuous geometric object and the finite record that an algorithm can update.

        ## Translation Guide

        | Textbook idea | Computational translation |
        | --- | --- |
        | Application frame | interpolating terrain heights from sample locations |
| Geometric problem | planar triangulations, Delaunay empty circle property, lifting, randomized construction |
| Algorithmic core | Delaunay triangulation as the projection of a lower convex hull |
| Data structures | triangle adjacency, circumcircle tests, conflict history |
| Visual model | delaunay |

        ## Route Through The Chapter

        1. compare arbitrary triangulations with max-min-angle Delaunay structure.
2. visualize the empty circumcircle condition.
3. lift points to a paraboloid and project lower faces.
4. trace randomized insertion and local flips.

        ## Visual Storyboard

        The visual sequence follows a fixed teaching rhythm. First, a concept diagram labels the input geometry and the claimed output. Second, an algorithm-state diagram shows the local decision that lets the algorithm avoid brute force. Third, a small metric table or JSON check records the invariant in numbers. Some chapters also add an interactive HTML artifact when rotation, ordering, or query movement is easier to inspect dynamically than in a single static figure.

        The source chapter's application is treated as motivation rather than as a turnkey software product. The notebook abstracts the application into a compact test instance, because a clean instance makes the correctness argument visible. For example, a map-overlay chapter should display the sweep status and then test it against brute-force intersections; a motion-planning chapter should display configuration obstacles and then test the route against collision predicates; a range-searching chapter should show the query rectangle and then compare the data-structure report with direct filtering.

        ## Worked Examples And Pitfalls

        The worked examples deliberately use small data. Small examples make it possible to see every event, cell, edge, or predicate outcome without trusting a black box. That is also how the notebook handles robustness. A geometric algorithm usually depends on predicates such as orientation, in-circle tests, distance comparison, visibility, or containment. If the predicate is unstable near degeneracy, the notebook displays a near-degenerate case and records a margin. The margin is not a proof of robust industrial arithmetic, but it tells the reader where exact arithmetic or symbolic perturbation would become relevant.

        The starred material for this chapter is treated as an advanced lens: framework for randomized algorithms. The notebook includes the starred idea as an optional computational extension whenever it changes the geometric picture. If it is mainly analytical, the extension becomes a small experiment: vary input size, random order, or query shape, then compare the measured state count with the stated asymptotic behavior. The point is to let the theorem leave a trace in an artifact, not to replace the proof with a picture.

        ## Implementation Lens

        The implementation is intentionally modest and inspectable. It favors plain arrays, short helper functions, and explicit predicates over a hidden production geometry kernel. That makes the notebook useful for learning: when a result changes, you can usually point to the exact comparison or update that changed it. The price is that these examples are teaching implementations, not industrial robust-geometry packages. The notebook therefore separates two claims. First, the displayed construction is checked on its own sample data. Second, the chapter explains what a complete implementation would still need for hostile inputs: exact predicates, careful event tie-breaking, balanced trees with stable keys, topology records, or certified numerical solvers.

        Read the final JSON artifact as a compact contract. It records the number of objects constructed, the key invariant, and the agreement with a direct check when a direct check is affordable. If you extend the notebook, keep that contract alive. Add a harder instance, then add a check that would fail if the geometric idea were misunderstood. For Delaunay Triangulations, a good extension keeps the same application frame but changes the input enough to stress triangle adjacency, circumcircle tests, conflict history. The goal is not to produce a large library in one notebook; the goal is to make the algorithm's finite state visible and falsifiable.

        ## Applied Lab

        The applied lab asks you to modify the sample instance while keeping the final checks true. Change a site, obstacle, segment, query window, or constraint. Then re-run the notebook and inspect which artifact changes first. If the answer changes but the invariant still holds, the model is doing useful work. If the invariant fails, the failure usually points to exactly the geometric assumption that the algorithm needs: general position, non-crossing input, convexity, sorted order, balanced refinement, or a complete visibility test.

        ## Sanity Checks To Read

        - no plotted Delaunay triangle contains another sample in its circumcircle.
- Euler count matches planar triangulation expectations.
- linear interpolation is continuous across shared edges.

        ## Takeaways

        By the end of the notebook, you should be able to explain the chapter without opening the PDF: what geometric object is being computed, which finite state represents it, why the state changes are local, which degeneracies matter, and what numerical or combinatorial check confirms the displayed result. That is the standard for this course: a chapter is complete when its prose, code, visuals, and checks together carry the computational geometry.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the CGAA book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, save_json
from utils.chapter_visuals import build_chapter_visuals, chapter_lab_summary

ARTIFACT_ROOT = BOOK_ROOT / "artifacts"


In [ ]:
source_span = json.loads('{\n  "number": 9,\n  "label": "Chapter 09",\n  "title": "Delaunay Triangulations",\n  "subtitle": "Height Interpolation",\n  "folder": "chapter-09-delaunay-triangulations",\n  "notebook": "09-delaunay-triangulations.ipynb",\n  "printed_pages": "191-218",\n  "pdf_pages": "202-229",\n  "focus": "planar triangulations, Delaunay empty circle property, lifting, randomized construction",\n  "application": "interpolating terrain heights from sample locations",\n  "algorithmic_core": "Delaunay triangulation as the projection of a lower convex hull",\n  "data_structures": "triangle adjacency, circumcircle tests, conflict history",\n  "starred": "framework for randomized algorithms",\n  "visual_kind": "delaunay",\n  "route": [\n    "compare arbitrary triangulations with max-min-angle Delaunay structure",\n    "visualize the empty circumcircle condition",\n    "lift points to a paraboloid and project lower faces",\n    "trace randomized insertion and local flips"\n  ],\n  "checks": [\n    "no plotted Delaunay triangle contains another sample in its circumcircle",\n    "Euler count matches planar triangulation expectations",\n    "linear interpolation is continuous across shared edges"\n  ],\n  "artifact_topic": "chapter-09"\n}')
CHAPTER_TOPIC = source_span["artifact_topic"]
CHAPTER_ARTIFACT_ROOT = ARTIFACT_ROOT / CHAPTER_TOPIC
CHAPTER_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
source_span


In [ ]:
visual_results = build_chapter_visuals(source_span, ARTIFACT_ROOT)
for item in visual_results["artifacts"]:
    display_artifact(BOOK_ROOT / item["relative_path"], width=760)
visual_results["summary"]


In [ ]:
lab_summary = chapter_lab_summary(source_span)
lab_summary


In [ ]:
final_sanity = {
    "chapter": source_span["label"],
    "title": source_span["title"],
    "source_span": {
        "printed_pages": source_span["printed_pages"],
        "pdf_pages": source_span["pdf_pages"],
    },
    "artifact_count": len(visual_results["artifacts"]),
    "visual_summary": visual_results["summary"],
    "lab_summary": lab_summary,
    "checks": source_span["checks"],
}
check_path = save_json(final_sanity, CHAPTER_ARTIFACT_ROOT / "checks" / "final-sanity.json")
required_paths = [BOOK_ROOT / item["relative_path"] for item in visual_results["artifacts"]]
required_paths.append(check_path)
assert_artifacts(required_paths)
final_sanity
